# Model 1: Random Forest Setup

Build two Question 1 modeling CSVs in the CSC 466 mixed-data format:

- `breed_traits_full.csv`: individual AKC trait scores plus coat categories.
- `breed_traits_mean.csv`: AKC trait-group mean scores plus coat categories.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

for base_dir in [Path.cwd(), *Path.cwd().parents]:
    data_dir = base_dir / "data"
    if data_dir.exists():
        PROJECT_ROOT = base_dir
        break
else:
    raise FileNotFoundError("Could not find project root with a data/ folder.")

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Load Cleaned Data

In [2]:
breed_traits = pd.read_csv(INTERIM_DIR / "breed_traits.csv")
breed_ranks = pd.read_csv(INTERIM_DIR / "breed_ranks.csv")

breed_traits.shape, breed_ranks.shape

((202, 17), (202, 14))

## Popularity Target

In [3]:
rank_cols = [f"{year} Rank" for year in range(2013, 2026)]
average_rank = np.nanmean(breed_ranks[rank_cols].to_numpy(dtype=float), axis=1)

breed_ranks["Average Rank"] = average_rank
breed_ranks["Popularity Tier"] = pd.cut(
    average_rank,
    bins=[0, 50, 125, np.inf],
    labels=["High Popularity", "Medium Popularity", "Low Popularity"],
    include_lowest=True,
)

## Trait Groups

In [4]:
trait_groups = {
    "Family Life": [
        "Affectionate With Family",
        "Good With Young Children",
        "Good With Other Dogs",
    ],
    "Physical": [
        "Shedding Level",
        "Coat Grooming Frequency",
        "Drooling Level",
    ],
    "Social": [
        "Openness To Strangers",
        "Playfulness Level",
        "Watchdog/Protective Nature",
        "Adaptability Level",

    ],
    "Personality": [
        "Trainability Level",
        "Energy Level",
        "Barking Level",
        "Mental Stimulation Needs",
    ],
}

numeric_trait_cols = [col for cols in trait_groups.values() for col in cols]
categorical_trait_cols = ["Coat Type", "Coat Length"]

breed_traits[numeric_trait_cols] = breed_traits[numeric_trait_cols].apply(pd.to_numeric, errors="coerce")

## Domain

In [5]:
def add_metadata(model_features, numeric_cols, class_col="Popularity Tier"):
    domain_codes = []
    for col in model_features.columns:
        if col == "Breed":
            domain_codes.append(-1)
        elif col in numeric_cols:
            domain_codes.append(0)
        else:
            domain_codes.append(model_features[col].nunique())

    domain_code_row = pd.DataFrame([domain_codes], columns=model_features.columns)
    class_variable_row = pd.DataFrame(
        [[class_col] + [""] * (len(model_features.columns) - 1)],
        columns=model_features.columns,
    )

    return pd.concat([domain_code_row, class_variable_row, model_features], ignore_index=True)


def write_csv(model_features, numeric_cols, output_name):
    output_path = PROCESSED_DIR / output_name
    model_features_with_metadata = add_metadata(model_features, numeric_cols)
    model_features_with_metadata.to_csv(output_path, index=False)
    return output_path, model_features_with_metadata

## Full Trait CSV

In [6]:
breed_traits_full = pd.concat(
    [
        breed_traits[["Breed"]],
        breed_traits[numeric_trait_cols],
        breed_traits[categorical_trait_cols],
        breed_ranks[["Popularity Tier"]],
    ],
    axis=1,
)

full_output_path, breed_traits_full_course = write_csv(
    breed_traits_full,
    numeric_trait_cols,
    "breed_traits_full.csv",
)

## Grouped Mean Trait CSV

In [7]:
breed_traits_mean = breed_traits[["Breed"]].copy()
mean_trait_cols = []

for group_name, cols in trait_groups.items():
    breed_traits_mean[group_name] = breed_traits[cols].mean(axis=1)
    mean_trait_cols.append(group_name)

breed_traits_mean = pd.concat(
    [
        breed_traits_mean,
        breed_traits[categorical_trait_cols],
        breed_ranks[["Popularity Tier"]],
    ],
    axis=1,
)

mean_output_path, breed_traits_mean_course = write_csv(
    breed_traits_mean,
    mean_trait_cols,
    "breed_traits_mean.csv",
)